In [ ]:
import time
PORT_STARTED = time.time()
PORT_DEADLINE = PORT_STARTED + 12 * 3600 - 600
PORT_MODE = 'package'  # Explicitly choose 'smoke' only for an authorized GPU check.
SMOKE_SECONDS = 900
EXPORT_SUBMISSION_PLACEHOLDER = True


In [ ]:
# Attributed source packaging only unless Kaggle competition rerun is active.
import base64, io, os, pathlib, subprocess, sys, zipfile
bundle = '...'
port_root = pathlib.Path('/kaggle/working/nvarc2026-carrier')
port_root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle))) as archive:
    archive.extractall(port_root)
command = [sys.executable, str(port_root / 'driver.py'), '--workdir', '/kaggle/working/nvarc2026-runtime']
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in ('1', 'true'):
    if PORT_MODE == 'smoke': raise RuntimeError('Smoke choice prohibited in competition rerun')
    command += ['--mode', 'competition', '--end-time', str(PORT_DEADLINE)]
else:
    if PORT_MODE not in ('package','smoke'): raise ValueError('Choose package or smoke')
    command += ['--mode', PORT_MODE]
    if PORT_MODE == 'package': command += ['--inspect-inputs']
    if PORT_MODE == 'package' and EXPORT_SUBMISSION_PLACEHOLDER:
        command += ['--export-placeholder']
    if PORT_MODE == 'smoke':
        command += ['--end-time', str(min(PORT_DEADLINE, PORT_STARTED+SMOKE_SECONDS)), '--smoke-seconds', str(SMOKE_SECONDS)]
# B1: pin Python hashing BEFORE starting the driver interpreter.
child_env = os.environ.copy()
child_env['PYTHONHASHSEED'] = '42'
subprocess.run(command, check=True, env=child_env)
